# **Random Forest with Differential Evolution:**

In [7]:
import pandas as pd
import numpy as np 

In [8]:
df= pd.read_csv('C:\\Users\\MyMachine\\Desktop\\Fuzzy-Logic-Based-Maternal-Mental-Health-Risk-Assessment\\Feature_Engineering\\hybrid_matrix.csv')

#### **🧬 What is Differential Evolution ($DE$)?**

Differential Evolution ($DE$) is a **`stochastic (randomized), population-based optimization algorithm`** that is used to find the `global minimum` (or `maximum`) of a continuous, non-linear, and often complex objective function. It belongs to the class of `Evolutionary Algorithms` ($EAs$), similar to `Genetic Algorithms` ($GAs$).

In our problem, $DE$ would be used to find the optimal values for the hundreds of fuzzy parameters ($a, b, c, d$ for trapezoidal, $c, \sigma$ for Gaussian, etc.) that define our fuzzy sets, such that a `Random Forest classifier` trained on the resulting ($\text{fuzzy\_data}$) achieves the best possible accuracy (or lowest error).

#### **Working of Differential Evolution:**

$DE$ operates on a population of potential solutions (vectors of fuzzy parameters) and iteratively improves them using simple vector arithmetic. It follows a cycle similar to other $EAs$: **Initialization $\rightarrow$ Mutation $\rightarrow$ Crossover $\rightarrow$ Selection.**

**1. Initialization:**   
* DE starts by generating a **`population`** of $N_P$ candidate solutions (vectors).

* Each candidate solution, $x_i$, is a long vector containing the values for all the fuzzy parameters we want to optimize (e.g., $x_i = [\text{RR\_Low } a, \text{RR\_Low } b, \dots, \text{Temp\_Fever } \sigma]$).

* These vectors are initialized randomly within the predefined bounds (minimum and maximum possible values) for each parameter.

**2. Mutation (The Core Idea: Difference):**

* For each candidate solution, $x_i$ (the "target vector"), DE creates a temporary **mutant vector**, $v_i$.

* This is done by randomly selecting three *other* distinct vectors from the current population, say $x_{r1}$, $x_{r2}$, and $x_{r3}$.

* The mutant vector is generated by **scaling the difference** between two of the vectors and adding it to the third:
    > $$\mathbf{v}_i = \mathbf{x}_{r1} + F \cdot (\mathbf{x}_{r2} - \mathbf{x}_{r3})$$

* $F$ is the **differential weight** (or mutation factor), a constant between 0 and 1. It controls the scale of the differential variation.

* **Intuition:** The term $(\mathbf{x}_{r2} - \mathbf{x}_{r3})$ represents a **direction and magnitude based on the current population's spread**. Adding this scaled difference to $\mathbf{x}_{r1}$ is a clever way to **explore the search space** in directions that have proven successful in the current generation.

**3. Crossover (Recombination):**

* After mutation, the mutant vector $\mathbf{v}_i$ is combined with the original target vector $\mathbf{x}_i$ to create a **trial vector**, $\mathbf{u}_i$.
* This is typically done using a **binomial (uniform) crossover**, where components of $\mathbf{u}_i$ are inherited from $\mathbf{v}_i$ or $\mathbf{x}_i$ based on a **crossover rate** ($CR$).

* If a random number is less than $CR$, the component comes from $\mathbf{v}_i$; otherwise, it comes from $\mathbf{x}_i$. This ensures that the trial vector inherits some properties from the successful variations while retaining some of the original solution's characteristics.

**4. Selection:**

* Finally, the **objective function** (your Random Forest accuracy/error) is evaluated for the new trial vector $\mathbf{u}_i$ and the original target vector $\mathbf{x}_i$.

* If the trial vector $\mathbf{u}_i$ yields a better result (e.g., higher accuracy for your Random Forest) than $\mathbf{x}_i$, then $\mathbf{x}_i$ is replaced by $\mathbf{u}_i$ in the next generation.
* If $\mathbf{x}_i$ is better, it is retained.

This process of mutation, crossover, and selection is repeated for many generations until a stopping criterion (e.g., maximum generations or negligible improvement) is met.

#### **🌍 Why $DE$ is a Global Optimization Algorithm?**

$DE$ is classified as a **`global optimization algorithm`** for the following key reasons:

1.  **`Population-Based Search`:**    
Unlike local search algorithms (like `gradient descent`) that follow a single path, $DE$ works with a large population of diverse candidate solutions simultaneously. This allows it to explore different regions of the search space in parallel.

2.  **`Difference Vector Mutuation`:**   
The core mutation step $\mathbf{x}_{r1} + F \cdot (\mathbf{x}_{r2} - \mathbf{x}_{r3})$ does not rely on local gradients or derivatives. The search direction is derived from the **`vector difference`** among random solutions, which is inherently designed to jump across valleys and escape local optima.

3.  **`Exploration vs. Exploitation`:**
    * **`Exploration (Global Search)`:** Large values of the scaling factor $F$ and the population size $N_P$ encourage the algorithm to make large, exploratory steps across the entire search space.

    * **`Exploitation (Local Search)`:** Small values of $F$ allow the algorithm to refine (exploit) the area around the best current solutions, ensuring convergence to the true optimum. This dynamic balance helps find the *absolute best* solution (global optimum).

#### **Intuitional Meaning:**

The core idea of Differential Evolution can be thought of as a **`dynamic, cooperative learning process`** among a group of search agents (the population vectors):

> "A search agent (target vector $x_i$) learns how to move towards a better solution by observing the **`difference in features`** between two *other* successful agents ($x_{r2}$ and $x_{r3}$) and applying that difference as a corrective action to a third agent ($x_{r1}$)."

It's a process of **`"trial and error guided by population diversity."`** The algorithm automatically uses the diversity of the current population to generate search steps, adapting the step size and direction based on how spread out the current solutions are.

#### **Geometric Meaning:**

Consider the entire optimization landscape as a high-dimensional surface where the height represents the error/fitness of the fuzzy parameters.

1.  **`The Difference Vector` ($\mathbf{x}_{r2} - \mathbf{x}_{r3}$):** This vector is a **`secant vector`** spanning two random points on the surface. Its direction and length estimate the current local variation and diversity within the population. It represents a **`probabilistic direction of potential improvement`.**

2.  **`The Addition` ($\mathbf{x}_{r1} + \dots$):** Adding the scaled difference vector to $\mathbf{x}_{r1}$ essentially defines a **new point $\mathbf{v}_i$ `that is a random linear combination of the current population points`.**

3.  **`Shifting the Search`:** Geometrically, $DE$ creates a random, translated parallelogram defined by the three random vectors. The mutant vector $\mathbf{v}_i$ is the fourth vertex of this shape. This geometric manipulation allows the new candidate to be placed far from the current best solutions, providing the **`"jump" necessary to escape local optima`** and continue searching for the global optimum.

#### **Why $DE$ Works Well for Fuzzy Parameter Optimization?**

**Fuzzy membership parameters often have:**   
   * many local minima
   * dependencies between parameters
   * non-linear effects
   * no closed-form derivative

**$DE$ handles these naturally:**    
   * It explores globally before narrowing.
   * It doesn’t require gradients.
   * It tunes all parameters jointly as a vector.
   * It avoids early convergence (a common issue in $GA$ or hill-climbing).

This is why $DE$ is one of the most common optimizers for fuzzy systems and control systems.

### **What are we optimizing?**    
> We're optimizing the **`parameters of the membership functions`** that define how crisp values are converted to fuzzy values. Our expert-driven fuzzification uses specific parameters (like `centers`, `spreads`, `boundaries`) that we want to improve using $DE$.

### **STEP- $0$: Understanding the Search Space (Parameter Encoding):**

Before Generation- $1$, we need to define what a `"solution"` looks like.

From our membership functions, we extract optimizable parameters:


````py 
# Here are all the fuzzy sets defined for each feture: 
# This is all expert defined: 

# ---------------------------------------------------
# -------------------------------------------------
#  1. Age: 

# Define  parameters for each fuzzy set 
# A. Young Mother (Left-Open Trapezoidal: [16, 16, 22, 28])
young_mother_params = [16, 16, 22, 28]

# B. Prime Motherhood (Triangular: [26, 30, 38])
prime_mother_params = [26, 30, 38]
 
# C. Advanced Maternal Age (Right-Open Trapezoidal: [36, 40, 44, 44])
advanced_age_params = [36, 40, 44, 44]


#---------------------------------------------------
# -------------------------------------------
# 2. Education Level: 

# Define the Discrete Fuzzification Map:

fuzzy_map_edu = {
    'No education':       {'mu_edu_low_risk': 1.0, 'mu_edu_medium_risk': 0.0, 'mu_edu_high_risk': 0.0},
    'Primary':            {'mu_edu_low_risk': 0.7, 'mu_edu_medium_risk': 0.3, 'mu_edu_high_risk': 0.0},
    'Secondary':          {'mu_edu_low_risk': 0.2, 'mu_edu_medium_risk': 0.8, 'mu_edu_high_risk': 0.0},
    'Higher':             {'mu_edu_low_risk': 0.0, 'mu_edu_medium_risk': 0.1, 'mu_edu_high_risk': 0.9},
}


# ---------------------------------------------------------
# ---------------------------------------------------------
# 3. wealth_index: 

# Define the Discrete Fuzzification Map
fuzzy_map_wealth = {
    'Poorest':  {'mu_wealth_high_poverty': 1.0, 'mu_wealth_medium_wealth': 0.0, 'mu_wealth_low_poverty': 0.0},
    'Poorer':   {'mu_wealth_high_poverty': 0.7, 'mu_wealth_medium_wealth': 0.3, 'mu_wealth_low_poverty': 0.0},
    'Middle':   {'mu_wealth_high_poverty': 0.3, 'mu_wealth_medium_wealth': 0.7, 'mu_wealth_low_poverty': 0.0},
    'Richer':   {'mu_wealth_high_poverty': 0.0, 'mu_wealth_medium_wealth': 0.3, 'mu_wealth_low_poverty': 0.7},
    'Richest':  {'mu_wealth_high_poverty': 0.0, 'mu_wealth_medium_wealth': 0.0, 'mu_wealth_low_poverty': 1.0},
}


# ---------------------------------------------------------------
# ----------------------------------------------------------------
# 4. monthly_household_income: 

MIN_INCOME = 8000.0
MAX_INCOME = 180000.0

# 1. Low Income (Trap): Full membership up to 20k, 0 at 45k
low_inc_params = [MIN_INCOME, 20000, 20000, 45000] 

# 2. Moderate Income (Tri): Starts 30k, Peaks 50k, Ends 100k
mod_inc_params = [30000, 50000, 100000]

# 3. High Income (Trap): 0 at 80k, Full membership from 120k
high_inc_params = [80000, 120000, MAX_INCOME, MAX_INCOME]


# --------------------------------------------------------
# -------------------------------------------------------
# 5. food_security: 

# Define the Discrete Fuzzification Map:
fuzzy_map_fs = {
    'Severely insecure':   {'mu_fs_high_risk': 1.0, 'mu_fs_mod_risk': 0.0, 'mu_fs_low_risk': 0.0},
    'Moderately insecure': {'mu_fs_high_risk': 0.8, 'mu_fs_mod_risk': 0.2, 'mu_fs_low_risk': 0.0},
    'Mildly insecure':     {'mu_fs_high_risk': 0.3, 'mu_fs_mod_risk': 0.7, 'mu_fs_low_risk': 0.0},
    'Secure':              {'mu_fs_high_risk': 0.0, 'mu_fs_mod_risk': 0.0, 'mu_fs_low_risk': 1.0},
}


# -----------------------------------------------------
# -------------------------------------------------
# 6. housing_quality: 

MIN_SCORE = 1.0
MAX_SCORE = 10.0
# Using steps of 0.1 for smooth interpolation
quality_universe = np.arange(MIN_SCORE, MAX_SCORE + 0.1, 0.1) 

# Define the Contextual Parameters for Fuzzy Sets: 

# A. Low Quality (Trap): Full membership up to 3, 0 at 6
low_hq_params = [1.0, 3.0, 3.0, 6.0] 

# B. Moderate Quality (Tri): Starts 4, Peaks 6 (Median), Ends 8
mod_hq_params = [4.0, 6.0, 8.0]

# C. High Quality (Trap): 0 at 7, Full membership from 9
high_hq_params = [7.0, 9.0, 10.0, 10.0]


# -----------------------------------------------------------
# -----------------------------------------------------------
# 7. financial_stress_level: 

MIN_SCORE = 0.0
MAX_SCORE = 10.0
# Using steps of 0.1 for smooth interpolation
stress_universe = np.arange(MIN_SCORE, MAX_SCORE + 0.1, 0.1) 

# Define the Parameters for Fuzzy Sets (based on Q1=3, Median=5, Q3=7)

# A. Low Stress (Trap): Full membership up to 2, 0 at 4
low_stress_params = [0.0, 2.0, 2.0, 4.0] 

# B. Moderate Stress (Tri): Starts 3 (Q1), Peaks 5 (Median), Ends 7 (Q3)
mod_stress_params = [3.0, 5.0, 7.0]

# C. High Stress (Trap): 0 at 6, Full membership from 8
high_stress_params = [6.0, 8.0, 10.0, 10.0]


# ---------------------------------------------------------
# ----------------------------------------------------------
# 8. parity: 

MIN_PARITY = 0.0
MAX_PARITY = 8.0
# Using steps of 0.1 for smooth interpolation
parity_universe = np.arange(MIN_PARITY, MAX_PARITY + 0.1, 0.1) 

# Define the Contextual Parameters for Fuzzy Sets

# A. Low Parity (Trap): Full membership up to 1, 0 at 3
low_p_params = [0.0, 1.0, 1.0, 3.0] 

# B. Optimal Parity (Tri): Starts 1, Peaks 2, Ends 4
opt_p_params = [1.0, 2.0, 4.0]

# C. High Parity (Trap): 0 at 3, Full membership from 5
high_p_params = [3.0, 5.0, 8.0, 8.0]


# --------------------------------------------------------------
# -------------------------------------------------------------
# 9. anc_visits:

MIN_VISITS = 0.0
MAX_VISITS = 12.0
# Using steps of 0.1 for smooth interpolation
visits_universe = np.arange(MIN_VISITS, MAX_VISITS + 0.1, 0.1) 

# Define the Contextual Parameters for Fuzzy Sets (based on WHO/Clinical Guidelines)

# A. Inadequate ANC (Trap): Full membership up to 3, 0 at 5
inad_anc_params = [0.0, 3.0, 3.0, 5.0] 

# B. Sufficient ANC (Tri): Starts 3, Peaks 5, Ends 8 (WHO previous standard)
suff_anc_params = [3.0, 5.0, 8.0]

# C. Optimal ANC (Trap): 0 at 7, Full membership from 9 (WHO current standard)
opt_anc_params = [7.0, 9.0, 12.0, 12.0]


# ---------------------------------------------------
# --------------------------------------------------
# 10. gestational_age: 

MIN_GA = 29.0
MAX_GA = 42.0
ga_universe = np.arange(MIN_GA, MAX_GA + 0.1, 0.1) 

# Define the Contextual Parameters for Fuzzy Sets (based on Clinical Standards)

# A. Preterm (Trap): Full membership up to 36, 0 at 38
preterm_params = [29.0, 36.0, 36.0, 38.0] 

# B. Optimal Term (Trap): Starts 37, Full membership 38-41, Ends 42
optimal_params = [37.0, 38.0, 41.0, 42.0]

# C. Late/Post-term (Trap): 0 at 40, Full membership from 42
late_post_params = [40.0, 42.0, 42.0, 42.0]


# ------------------------------------------------------------
# --------------------------------------------------------------
# 11. maternal_age_at_first_birth: 

MIN_AGE = 15.0
MAX_AGE = 44.0
age_universe = np.arange(MIN_AGE, MAX_AGE + 0.1, 0.1) 

# Define the Contextual Parameters for Fuzzy Sets (based on Clinical Standards) ---

# A. Early Motherhood (Trap): Full membership up to 19, 0 at 22
early_m_params = [15.0, 19.0, 19.0, 22.0] 

# B. Optimal Age (Trap): Starts 20, Full membership 21-33, Ends 35
optimal_age_params = [20.0, 21.0, 33.0, 35.0]

# C. Advanced Age (Trap): 0 at 34, Full membership from 35
advanced_a_params = [34.0, 35.0, 44.0, 44.0]


# -----------------------------------------------------
# -------------------------------------------------------
# 12. partner_support_score: 

MIN_SCORE = 0.0
MAX_SCORE = 10.0
score_universe = np.arange(MIN_SCORE, MAX_SCORE + 0.1, 0.1) 

# Define the Contextual Parameters for Fuzzy Sets

# A. Low Support (Trap): Full membership up to 4, 0 at 6
low_s_params = [0.0, 4.0, 4.0, 6.0] 

# B. Moderate Support (Tri): Starts 4, Peaks 6, Ends 8
mod_s_params = [4.0, 6.0, 8.0]

# C. High Support (Trap): 0 at 7, Full membership from 9
high_s_params = [7.0, 9.0, 10.0, 10.0]


# ---------------------------------------------------
# ----------------------------------------------------
# 13. family_support_score: 

MIN_SCORE = 0.0
MAX_SCORE = 10.0
score_universe = np.arange(MIN_SCORE, MAX_SCORE + 0.1, 0.1) 

# Define the Contextual Parameters for Fuzzy Sets

# A. Inadequate Support (Trap): Full membership up to 4, 0 at 6
inad_f_params = [0.0, 4.0, 4.0, 6.0] 

# B. Moderate Support (Tri): Starts 4, Peaks 6, Ends 8
mod_f_params = [4.0, 6.0, 8.0]

# C. Strong Support (Trap): 0 at 7, Full membership from 9
strong_f_params = [7.0, 9.0, 10.0, 10.0]


# --------------------------------------------------------------
# --------------------------------------------------------------
# 14. domestic_violence_exposure: 

# Define the Discrete Fuzzification Map: 
fuzzy_map_dv = {
    'Often':       {'mu_dv_high_risk': 1.0, 'mu_dv_mod_risk': 0.0, 'mu_dv_no_risk': 0.0},
    'Sometimes':   {'mu_dv_high_risk': 0.9, 'mu_dv_mod_risk': 0.1, 'mu_dv_no_risk': 0.0},
    'Rarely':      {'mu_dv_high_risk': 0.3, 'mu_dv_mod_risk': 0.7, 'mu_dv_no_risk': 0.0},
    'Never':       {'mu_dv_high_risk': 0.0, 'mu_dv_mod_risk': 0.0, 'mu_dv_no_risk': 1.0},
}


# ------------------------------------------------------
# -----------------------------------------------------
# 15. social_isolation_score: 

MIN_SCORE = 0.0
MAX_SCORE = 10.0
score_universe = np.arange(MIN_SCORE, MAX_SCORE + 0.1, 0.1) 

# Define the Contextual Parameters for Fuzzy Sets

# A. Low Isolation (Trap): Full membership up to 2, 0 at 4
low_iso_params = [0.0, 2.0, 2.0, 4.0] 

# B. Moderate Isolation (Tri): Starts 3, Peaks 4.5, Ends 7
mod_iso_params = [3.0, 4.5, 7.0]

# C. High Isolation (Trap): 0 at 6, Full membership from 8
high_iso_params = [6.0, 8.0, 10.0, 10.0]


# ---------------------------------------------------------------
# ---------------------------------------------------------------
# 16. sleep_quality: 

MIN_SCORE = 1.0
MAX_SCORE = 10.0
score_universe = np.arange(MIN_SCORE, MAX_SCORE + 0.1, 0.1) 

# Define the Contextual Parameters for Fuzzy Sets

# A. Poor Sleep (Trap): Full membership up to 4, 0 at 6
poor_s_params = [1.0, 4.0, 4.0, 6.0] 

# B. Moderate Sleep (Tri): Starts 4, Peaks 6.5, Ends 8
mod_s_params = [4.0, 6.5, 8.0]

# C. Good Sleep (Trap): 0 at 7, Full membership from 9
good_s_params = [7.0, 9.0, 10.0, 10.0]


# --------------------------------------------------------
# -------------------------------------------------------
# 17. stressful_life_events: 

MIN_SCORE = 0.0
MAX_SCORE = 5.0
score_universe = np.arange(MIN_SCORE, MAX_SCORE + 0.1, 0.1) 

# Define the Contextual Parameters for Fuzzy Sets

# A. Low Stress Load (Trap): Full membership at 0, 0 at 1.5
low_sl_params = [0.0, 0.0, 0.0, 1.5] 

# B. Moderate Stress Load (Tri): Starts 0.5, Peaks 2, Ends 3.5
mod_sl_params = [0.5, 2.0, 3.5]

# C. High Stress Load (Trap): 0 at 3, Full membership from 4
high_sl_params = [3.0, 4.0, 5.0, 5.0]


# -----------------------------------------
# -------------------------------------------
# 18. distance_to_health_facility: 

MIN_DIST = 0.5
MAX_DIST = 50.0
dist_universe = np.arange(MIN_DIST, MAX_DIST + 0.5, 0.5) 

# Define the Contextual Parameters for Fuzzy Sets

# A. Easy Access (Trap): Full membership up to 2 km, 0 at 5 km
easy_a_params = [0.5, 2.0, 2.0, 5.0] 

# B. Moderate Access (Tri): Starts 3 km, Peaks 6 km, Ends 10 km
mod_a_params = [3.0, 6.0, 10.0]

# C. Difficult Access (Trap): 0 at 8 km, Full membership from 15 km
diff_a_params = [8.0, 15.0, 50.0, 50.0]
``` 

#### **Continuous Features (Trapezoidal/Triangular Membership Functions):**

**These features have geometric parameters that can be optimized:**

1. **Age (3 fuzzy sets)**
   - Young Mother (Trap): [16, 16, 22, 28] → **2 params** (22, 28)
   - Prime Motherhood (Tri): [26, 30, 38] → **3 params** (26, 30, 38)
   - Advanced Age (Trap): [36, 40, 44, 44] → **2 params** (36, 40)
   - **Subtotal: 7 parameters**

2. **Monthly Household Income (3 fuzzy sets)**
   - Low Income (Trap): [8000, 20000, 20000, 45000] → **3 params** (20000, 20000, 45000)
   - Moderate Income (Tri): [30000, 50000, 100000] → **3 params**
   - High Income (Trap): [80000, 120000, 180000, 180000] → **2 params** (80000, 120000)
   - **Subtotal: 8 parameters**

3. **Housing Quality (3 fuzzy sets)**
   - Low (Trap): [1.0, 3.0, 3.0, 6.0] → **3 params** (3.0, 3.0, 6.0)
   - Moderate (Tri): [4.0, 6.0, 8.0] → **3 params**
   - High (Trap): [7.0, 9.0, 10.0, 10.0] → **2 params** (7.0, 9.0)
   - **Subtotal: 8 parameters**

4. **Financial Stress Level (3 fuzzy sets)**
   - Low (Trap): [0.0, 2.0, 2.0, 4.0] → **3 params** (2.0, 2.0, 4.0)
   - Moderate (Tri): [3.0, 5.0, 7.0] → **3 params**
   - High (Trap): [6.0, 8.0, 10.0, 10.0] → **2 params** (6.0, 8.0)
   - **Subtotal: 8 parameters**

5. **Parity (3 fuzzy sets)**
   - Low (Trap): [0.0, 1.0, 1.0, 3.0] → **3 params** (1.0, 1.0, 3.0)
   - Optimal (Tri): [1.0, 2.0, 4.0] → **3 params**
   - High (Trap): [3.0, 5.0, 8.0, 8.0] → **2 params** (3.0, 5.0)
   - **Subtotal: 8 parameters**

6. **ANC Visits (3 fuzzy sets)**
   - Inadequate (Trap): [0.0, 3.0, 3.0, 5.0] → **3 params** (3.0, 3.0, 5.0)
   - Sufficient (Tri): [3.0, 5.0, 8.0] → **3 params**
   - Optimal (Trap): [7.0, 9.0, 12.0, 12.0] → **2 params** (7.0, 9.0)
   - **Subtotal: 8 parameters**

7. **Gestational Age (3 fuzzy sets)**
   - Preterm (Trap): [29.0, 36.0, 36.0, 38.0] → **3 params** (36.0, 36.0, 38.0)
   - Optimal (Trap): [37.0, 38.0, 41.0, 42.0] → **4 params**
   - Late/Post (Trap): [40.0, 42.0, 42.0, 42.0] → **2 params** (40.0, 42.0)
   - **Subtotal: 9 parameters**

8. **Maternal Age at First Birth (3 fuzzy sets)**
   - Early (Trap): [15.0, 19.0, 19.0, 22.0] → **3 params** (19.0, 19.0, 22.0)
   - Optimal (Trap): [20.0, 21.0, 33.0, 35.0] → **4 params**
   - Advanced (Trap): [34.0, 35.0, 44.0, 44.0] → **2 params** (34.0, 35.0)
   - **Subtotal: 9 parameters**

9. **Partner Support Score (3 fuzzy sets)**
   - Low (Trap): [0.0, 4.0, 4.0, 6.0] → **3 params** (4.0, 4.0, 6.0)
   - Moderate (Tri): [4.0, 6.0, 8.0] → **3 params**
   - High (Trap): [7.0, 9.0, 10.0, 10.0] → **2 params** (7.0, 9.0)
   - **Subtotal: 8 parameters**

10. **Family Support Score (3 fuzzy sets)**
    - Inadequate (Trap): [0.0, 4.0, 4.0, 6.0] → **3 params** (4.0, 4.0, 6.0)
    - Moderate (Tri): [4.0, 6.0, 8.0] → **3 params**
    - Strong (Trap): [7.0, 9.0, 10.0, 10.0] → **2 params** (7.0, 9.0)
    - **Subtotal: 8 parameters**

11. **Social Isolation Score (3 fuzzy sets)**
    - Low (Trap): [0.0, 2.0, 2.0, 4.0] → **3 params** (2.0, 2.0, 4.0)
    - Moderate (Tri): [3.0, 4.5, 7.0] → **3 params**
    - High (Trap): [6.0, 8.0, 10.0, 10.0] → **2 params** (6.0, 8.0)
    - **Subtotal: 8 parameters**

12. **Sleep Quality (3 fuzzy sets)**
    - Poor (Trap): [1.0, 4.0, 4.0, 6.0] → **3 params** (4.0, 4.0, 6.0)
    - Moderate (Tri): [4.0, 6.5, 8.0] → **3 params**
    - Good (Trap): [7.0, 9.0, 10.0, 10.0] → **2 params** (7.0, 9.0)
    - **Subtotal: 8 parameters**

13. **Stressful Life Events (3 fuzzy sets)**
    - Low (Trap): [0.0, 0.0, 0.0, 1.5] → **1 param** (1.5)
    - Moderate (Tri): [0.5, 2.0, 3.5] → **3 params**
    - High (Trap): [3.0, 4.0, 5.0, 5.0] → **2 params** (3.0, 4.0)
    - **Subtotal: 6 parameters**

14. **Distance to Health Facility (3 fuzzy sets)**
    - Easy (Trap): [0.5, 2.0, 2.0, 5.0] → **3 params** (2.0, 2.0, 5.0)
    - Moderate (Tri): [3.0, 6.0, 10.0] → **3 params**
    - Difficult (Trap): [8.0, 15.0, 50.0, 50.0] → **2 params** (8.0, 15.0)
    - **Subtotal: 8 parameters**

#### **Discrete Features (Membership Degree Maps):**

**These features have membership degrees that can be optimized:**

15. **Education Level (4 categories × 3 fuzzy sets)** → **12 parameters**

16. **Wealth Index (5 categories × 3 fuzzy sets)** → **15 parameters**

17. **Food Security (4 categories × 3 fuzzy sets)** → **12 parameters**

18. **Domestic Violence (4 categories × 3 fuzzy sets)** → **12 parameters**

## **Setting Bounds:**

In [ ]:
import numpy as np

# Create initial solution vector: 
# SOLUTION VECTOR STRUCTURE (INITIAL VALUES FROM EXPERT KNOWLEDGE)

def create_initial_solution_vector():
    """
    Creates the initial solution vector with expert-defined parameter values.
    This represents the starting point for DE optimization.
    
    Total: 150 parameters (99 continuous + 51 discrete)
    """
    solution = []
    
    # ===========
    # CONTINUOUS FEATURES (99 parameters)
    # ===========
    
    # 1. AGE [Domain: 16-44] - 7 params
    # Young Mother trap [16, 16, c, d]
    solution.extend([22, 28])  # c, d
    # Prime Motherhood tri [a, b, c]
    solution.extend([26, 30, 38])  # a, b, c
    # Advanced Age trap [a, b, 44, 44]
    solution.extend([36, 40])  # a, b
    
    # 2. MONTHLY HOUSEHOLD INCOME [Domain: 8000-180000] - 7 params
    # Low Income trap [8000, b, b, d] where b=c
    solution.extend([20000, 45000])  # b, d
    # Moderate Income tri [a, b, c]
    solution.extend([30000, 50000, 100000])  # a, b, c
    # High Income trap [a, b, 180000, 180000]
    solution.extend([80000, 120000])  # a, b
    
    # 3. HOUSING QUALITY [Domain: 1-10] - 7 params
    # Low Quality trap [1.0, b, b, d] where b=c
    solution.extend([3.0, 6.0])  # b, d
    # Moderate Quality tri [a, b, c]
    solution.extend([4.0, 6.0, 8.0])  # a, b, c
    # High Quality trap [a, b, 10.0, 10.0]
    solution.extend([7.0, 9.0])  # a, b
    
    # 4. FINANCIAL STRESS LEVEL [Domain: 0-10] - 7 params
    # Low Stress trap [0.0, b, b, d] where b=c
    solution.extend([2.0, 4.0])  # b, d
    # Moderate Stress tri [a, b, c]
    solution.extend([3.0, 5.0, 7.0])  # a, b, c
    # High Stress trap [a, b, 10.0, 10.0]
    solution.extend([6.0, 8.0])  # a, b
    
    # 5. PARITY [Domain: 0-8] - 7 params
    # Low Parity trap [0.0, b, b, d] where b=c
    solution.extend([1.0, 3.0])  # b, d
    # Optimal Parity tri [a, b, c]
    solution.extend([1.0, 2.0, 4.0])  # a, b, c
    # High Parity trap [a, b, 8.0, 8.0]
    solution.extend([3.0, 5.0])  # a, b
    
    # 6. ANC VISITS [Domain: 0-12] - 7 params
    # Inadequate ANC trap [0.0, b, b, d] where b=c
    solution.extend([3.0, 5.0])  # b, d
    # Sufficient ANC tri [a, b, c]
    solution.extend([3.0, 5.0, 8.0])  # a, b, c
    # Optimal ANC trap [a, b, 12.0, 12.0]
    solution.extend([7.0, 9.0])  # a, b
    
    # 7. GESTATIONAL AGE [Domain: 29-42] - 8 params
    # Preterm trap [29.0, b, b, d] where b=c
    solution.extend([36.0, 38.0])  # b, d
    # Optimal Term trap [a, b, c, d] - full trapezoid
    solution.extend([37.0, 38.0, 41.0, 42.0])  # a, b, c, d
    # Late/Post-term trap [a, b, 42.0, 42.0]
    solution.extend([40.0, 42.0])  # a, b
    
    # 8. MATERNAL AGE AT FIRST BIRTH [Domain: 15-44] - 8 params
    # Early Motherhood trap [15.0, b, b, d] where b=c
    solution.extend([19.0, 22.0])  # b, d
    # Optimal Age trap [a, b, c, d] - full trapezoid
    solution.extend([20.0, 21.0, 33.0, 35.0])  # a, b, c, d
    # Advanced Age trap [a, b, 44.0, 44.0]
    solution.extend([34.0, 35.0])  # a, b
    
    # 9. PARTNER SUPPORT SCORE [Domain: 0-10] - 7 params
    # Low Support trap [0.0, b, b, d] where b=c
    solution.extend([4.0, 6.0])  # b, d
    # Moderate Support tri [a, b, c]
    solution.extend([4.0, 6.0, 8.0])  # a, b, c
    # High Support trap [a, b, 10.0, 10.0]
    solution.extend([7.0, 9.0])  # a, b
    
    # 10. FAMILY SUPPORT SCORE [Domain: 0-10] - 7 params
    # Inadequate Support trap [0.0, b, b, d] where b=c
    solution.extend([4.0, 6.0])  # b, d
    # Moderate Support tri [a, b, c]
    solution.extend([4.0, 6.0, 8.0])  # a, b, c
    # Strong Support trap [a, b, 10.0, 10.0]
    solution.extend([7.0, 9.0])  # a, b
    
    # 11. SOCIAL ISOLATION SCORE [Domain: 0-10] - 7 params
    # Low Isolation trap [0.0, b, b, d] where b=c
    solution.extend([2.0, 4.0])  # b, d
    # Moderate Isolation tri [a, b, c]
    solution.extend([3.0, 4.5, 7.0])  # a, b, c
    # High Isolation trap [a, b, 10.0, 10.0]
    solution.extend([6.0, 8.0])  # a, b
    
    # 12. SLEEP QUALITY [Domain: 1-10] - 7 params
    # Poor Sleep trap [1.0, b, b, d] where b=c
    solution.extend([4.0, 6.0])  # b, d
    # Moderate Sleep tri [a, b, c]
    solution.extend([4.0, 6.5, 8.0])  # a, b, c
    # Good Sleep trap [a, b, 10.0, 10.0]
    solution.extend([7.0, 9.0])  # a, b
    
    # 13. STRESSFUL LIFE EVENTS [Domain: 0-5] - 6 params
    # Low Stress Load trap [0.0, 0.0, 0.0, d]
    solution.extend([1.5])  # d only
    # Moderate Stress Load tri [a, b, c]
    solution.extend([0.5, 2.0, 3.5])  # a, b, c
    # High Stress Load trap [a, b, 5.0, 5.0]
    solution.extend([3.0, 4.0])  # a, b
    
    # 14. DISTANCE TO HEALTH FACILITY [Domain: 0.5-50] - 7 params
    # Easy Access trap [0.5, b, b, d] where b=c
    solution.extend([2.0, 5.0])  # b, d
    # Moderate Access tri [a, b, c]
    solution.extend([3.0, 6.0, 10.0])  # a, b, c
    # Difficult Access trap [a, b, 50.0, 50.0]
    solution.extend([8.0, 15.0])  # a, b
    
    # ===========
    # DISCRETE FEATURES (51 parameters)
    # ===========
    
    # 15. EDUCATION LEVEL - 12 params (4 categories × 3 membership degrees)
    solution.extend([
        1.0, 0.0, 0.0,  # No education: [low_risk, medium_risk, high_risk]
        0.7, 0.3, 0.0,  # Primary
        0.2, 0.8, 0.0,  # Secondary
        0.0, 0.1, 0.9   # Higher
    ])
    
    # 16. WEALTH INDEX - 15 params (5 categories × 3 membership degrees)
    solution.extend([
        1.0, 0.0, 0.0,  # Poorest: [high_poverty, medium_wealth, low_poverty]
        0.7, 0.3, 0.0,  # Poorer
        0.3, 0.7, 0.0,  # Middle
        0.0, 0.3, 0.7,  # Richer
        0.0, 0.0, 1.0   # Richest
    ])
    
    # 17. FOOD SECURITY - 12 params (4 categories × 3 membership degrees)
    solution.extend([
        1.0, 0.0, 0.0,  # Severely insecure: [high_risk, mod_risk, low_risk]
        0.8, 0.2, 0.0,  # Moderately insecure
        0.3, 0.7, 0.0,  # Mildly insecure
        0.0, 0.0, 1.0   # Secure
    ])
    
    # 18. DOMESTIC VIOLENCE EXPOSURE - 12 params (4 categories × 3 membership degrees)
    solution.extend([
        1.0, 0.0, 0.0,  # Often: [high_risk, mod_risk, no_risk]
        0.9, 0.1, 0.0,  # Sometimes
        0.3, 0.7, 0.0,  # Rarely
        0.0, 0.0, 1.0   # Never
    ])
    
    return np.array(solution)

We'll also need to define bounds that respect:
1. **`Ordering constraints`** (e.g., for triangular: $a < b < c$)
2. **`Domain limits`** (e.g., age: $16-44$, income: $8000-180000$)
3. **`Membership degree constraints`** (sum to $≤ 1.0$ for discrete features)

**Total Dimension: D: $150$ parameters.**

If we have **`150 fuzzy parameters`** (e.g., $a, b, c, ....,  \sigma$ values) across all our membership functions, the **`solution vector`** (or `candidate solution`) is simply a single vector of length $150$ that contains one specific value for each of those parameters. The **`sole use`** of this solution vector in the optimization process (Differential Evolution process) is to **`define a complete set of membership functions`**; this set is then used to **fuzzify the raw data** into a feature matrix, which is subsequently used to train a **`machine learning model (e.g., Random Forest)`**, allowing the algorithm to calculate the model's performance (e.g., accuracy) and thereby evaluate the **`quality`** of that particular vector of fuzzy parameters.

In [ ]:
# BOUNDS DEFINITION FOR DIFFERENTIAL EVOLUTION

def create_de_bounds():
    """
    Creates bounds for all 150 parameters.
    Returns: numpy array of shape (150, 2) with [lower_bound, upper_bound]
    """
    bounds = []
    
    # ================
    # CONTINUOUS FEATURES BOUNDS (99 parameters)
    # ================
    
    # 1. AGE - 7 params
    bounds.extend([
        [18, 26],      # Young c
        [24, 32],      # Young d
        [22, 28],      # Prime a
        [28, 32],      # Prime b
        [32, 40],      # Prime c
        [32, 38],      # Advanced a
        [36, 42]       # Advanced b
    ])
    
    # 2. INCOME - 7 params
    bounds.extend([
        [15000, 25000],    # Low b
        [35000, 55000],    # Low d
        [25000, 40000],    # Moderate a
        [40000, 60000],    # Moderate b
        [80000, 120000],   # Moderate c
        [70000, 90000],    # High a
        [100000, 140000]   # High b
    ])
    
    # 3. HOUSING QUALITY - 7 params
    bounds.extend([
        [2.0, 4.0],    # Low b
        [5.0, 7.0],    # Low d
        [3.0, 5.0],    # Moderate a
        [5.0, 7.0],    # Moderate b
        [7.0, 9.0],    # Moderate c
        [6.0, 8.0],    # High a
        [8.0, 9.5]     # High b
    ])
    
    # 4. FINANCIAL STRESS - 7 params
    bounds.extend([
        [1.0, 3.0],    # Low b
        [3.0, 5.0],    # Low d
        [2.0, 4.0],    # Moderate a
        [4.0, 6.0],    # Moderate b
        [6.0, 8.0],    # Moderate c
        [5.0, 7.0],    # High a
        [7.0, 9.0]     # High b
    ])
    
    # 5. PARITY - 7 params
    bounds.extend([
        [0.5, 1.5],    # Low b
        [2.0, 4.0],    # Low d
        [0.5, 1.5],    # Optimal a
        [1.5, 2.5],    # Optimal b
        [3.0, 5.0],    # Optimal c
        [2.0, 4.0],    # High a
        [4.0, 6.0]     # High b
    ])
    
    # 6. ANC VISITS - 7 params
    bounds.extend([
        [2.0, 4.0],    # Inadequate b
        [4.0, 6.0],    # Inadequate d
        [2.0, 4.0],    # Sufficient a
        [4.0, 6.0],    # Sufficient b
        [7.0, 9.0],    # Sufficient c
        [6.0, 8.0],    # Optimal a
        [8.0, 10.0]    # Optimal b
    ])
    
    # 7. GESTATIONAL AGE - 8 params
    bounds.extend([
        [34.0, 37.0],  # Preterm b
        [37.0, 39.0],  # Preterm d
        [36.0, 38.0],  # Optimal a
        [37.0, 39.0],  # Optimal b
        [40.0, 41.5],  # Optimal c
        [41.0, 42.0],  # Optimal d
        [39.0, 41.0],  # Late a
        [41.0, 42.0]   # Late b
    ])
    
    # 8. MATERNAL AGE FIRST BIRTH - 8 params
    bounds.extend([
        [17.0, 20.0],  # Early b
        [20.0, 24.0],  # Early d
        [18.0, 22.0],  # Optimal a
        [20.0, 24.0],  # Optimal b
        [30.0, 35.0],  # Optimal c
        [33.0, 37.0],  # Optimal d
        [32.0, 36.0],  # Advanced a
        [34.0, 38.0]   # Advanced b
    ])
    
    # 9. PARTNER SUPPORT - 7 params
    bounds.extend([
        [3.0, 5.0],    # Low b
        [5.0, 7.0],    # Low d
        [3.0, 5.0],    # Moderate a
        [5.0, 7.0],    # Moderate b
        [7.0, 9.0],    # Moderate c
        [6.0, 8.0],    # High a
        [8.0, 9.5]     # High b
    ])
    
    # 10. FAMILY SUPPORT - 7 params
    bounds.extend([
        [3.0, 5.0],    # Inadequate b
        [5.0, 7.0],    # Inadequate d
        [3.0, 5.0],    # Moderate a
        [5.0, 7.0],    # Moderate b
        [7.0, 9.0],    # Moderate c
        [6.0, 8.0],    # Strong a
        [8.0, 9.5]     # Strong b
    ])
    
    # 11. SOCIAL ISOLATION - 7 params
    bounds.extend([
        [1.0, 3.0],    # Low b
        [3.0, 5.0],    # Low d
        [2.0, 4.0],    # Moderate a
        [4.0, 5.5],    # Moderate b
        [6.0, 8.0],    # Moderate c
        [5.0, 7.0],    # High a
        [7.0, 9.0]     # High b
    ])
    
    # 12. SLEEP QUALITY - 7 params
    bounds.extend([
        [3.0, 5.0],    # Poor b
        [5.0, 7.0],    # Poor d
        [3.0, 5.0],    # Moderate a
        [5.5, 7.5],    # Moderate b
        [7.0, 9.0],    # Moderate c
        [6.0, 8.0],    # Good a
        [8.0, 9.5]     # Good b
    ])
    
    # 13. STRESSFUL LIFE EVENTS - 6 params
    bounds.extend([
        [1.0, 2.0],    # Low d
        [0.3, 0.8],    # Moderate a
        [1.5, 2.5],    # Moderate b
        [3.0, 4.0],    # Moderate c
        [2.5, 3.5],    # High a
        [3.5, 4.5]     # High b
    ])
    
    # 14. DISTANCE TO FACILITY - 7 params
    bounds.extend([
        [1.5, 2.5],    # Easy b
        [4.0, 6.0],    # Easy d
        [2.5, 4.0],    # Moderate a
        [5.0, 7.0],    # Moderate b
        [8.0, 12.0],   # Moderate c
        [7.0, 10.0],   # Difficult a
        [12.0, 18.0]   # Difficult b
    ])
    
    # ===========================================
    # DISCRETE FEATURES BOUNDS (51 parameters)
    # All membership degrees bounded to [0.0, 1.0]
    # ===========================================
    
    # Education (12) + Wealth (15) + Food Security (12) + Domestic Violence (12)
    for _ in range(51):
        bounds.append([0.0, 1.0])
    
    return np.array(bounds)

In [20]:
# Create solution vector and bounds
initial_solution = create_initial_solution_vector()
bounds = create_de_bounds()

In [22]:
len(initial_solution)

150

In [24]:
len(bounds)

150

In [25]:
# Display results
print("SOLUTION VECTOR STRUCTURE:")
print(f"Total parameters: {len(initial_solution)}")
print(f"\nFirst 20 parameters (continuous features):")
print(initial_solution[:20])
print(f"\nLast 12 parameters (discrete features - Education):")
print(initial_solution[101:113])

SOLUTION VECTOR STRUCTURE:
Total parameters: 150

First 20 parameters (continuous features):
[2.2e+01 2.8e+01 2.6e+01 3.0e+01 3.8e+01 3.6e+01 4.0e+01 2.0e+04 4.5e+04
 3.0e+04 5.0e+04 1.0e+05 8.0e+04 1.2e+05 3.0e+00 6.0e+00 4.0e+00 6.0e+00
 8.0e+00 7.0e+00]

Last 12 parameters (discrete features - Education):
[0.  0.7 0.3 0.  0.2 0.8 0.  0.  0.1 0.9 1.  0. ]


In [26]:
print("BOUNDS ARRAY:")
print(f"Bounds shape: {bounds.shape}")
print(f"\nFirst 10 bounds (Age + Income start):")
for i in range(10):
    print(f"  Param {i}: [{bounds[i,0]:>10.1f}, {bounds[i,1]:>10.1f}]")

BOUNDS ARRAY:
Bounds shape: (150, 2)

First 10 bounds (Age + Income start):
  Param 0: [      18.0,       26.0]
  Param 1: [      24.0,       32.0]
  Param 2: [      22.0,       28.0]
  Param 3: [      28.0,       32.0]
  Param 4: [      32.0,       40.0]
  Param 5: [      32.0,       38.0]
  Param 6: [      36.0,       42.0]
  Param 7: [   15000.0,    25000.0]
  Param 8: [   35000.0,    55000.0]
  Param 9: [   25000.0,    40000.0]


In [27]:
# Detailed parameter breakdown
print("PARAMETER COUNT BREAKDOWN:")
continuous_counts = [7, 7, 7, 7, 7, 7, 8, 8, 7, 7, 7, 7, 6, 7]
features = [
    'Age', 'Monthly Income', 'Housing Quality', 'Financial Stress', 
    'Parity', 'ANC Visits', 'Gestational Age', 'Maternal Age First Birth',
    'Partner Support', 'Family Support', 'Social Isolation', 
    'Sleep Quality', 'Stressful Life Events', 'Distance to Facility'
]

idx = 0
for i, (feat, count) in enumerate(zip(features, continuous_counts), 1):
    print(f"{i:2d}. {feat:30s}: params {idx:3d}-{idx+count-1:3d} ({count} params)")
    idx += count

print(f"\n{'Continuous Total':>33s}: {sum(continuous_counts)} params (0-{sum(continuous_counts)-1})")
print(f"{'Discrete Total':>33s}: 51 params ({sum(continuous_counts)}-{sum(continuous_counts)+50})")
print(f"{'GRAND TOTAL':>33s}: {sum(continuous_counts) + 51} params")

PARAMETER COUNT BREAKDOWN:
 1. Age                           : params   0-  6 (7 params)
 2. Monthly Income                : params   7- 13 (7 params)
 3. Housing Quality               : params  14- 20 (7 params)
 4. Financial Stress              : params  21- 27 (7 params)
 5. Parity                        : params  28- 34 (7 params)
 6. ANC Visits                    : params  35- 41 (7 params)
 7. Gestational Age               : params  42- 49 (8 params)
 8. Maternal Age First Birth      : params  50- 57 (8 params)
 9. Partner Support               : params  58- 64 (7 params)
10. Family Support                : params  65- 71 (7 params)
11. Social Isolation              : params  72- 78 (7 params)
12. Sleep Quality                 : params  79- 85 (7 params)
13. Stressful Life Events         : params  86- 91 (6 params)
14. Distance to Facility          : params  92- 98 (7 params)

                 Continuous Total: 99 params (0-98)
                   Discrete Total: 51 params (99-149

In [28]:
# Verification
print("VERIFICATION:")
assert len(initial_solution) == 150, f"Solution vector: Expected 150, got {len(initial_solution)}"
assert bounds.shape == (150, 2), f"Bounds: Expected (150, 2), got {bounds.shape}"
assert np.all(bounds[:, 0] <= bounds[:, 1]), "All lower bounds must be <= upper bounds"

# Verify initial solution is within bounds
within_bounds = np.all((initial_solution >= bounds[:, 0]) & (initial_solution <= bounds[:, 1]))
assert within_bounds, "Initial solution must be within bounds"

print("Solution vector length: 150")
print("Bounds shape: (150, 2)")
print("All lower bounds <= upper bounds")
print("Initial solution within bounds")
print("SUCCESS: All structures are correctly defined!")

VERIFICATION:
Solution vector length: 150
Bounds shape: (150, 2)
All lower bounds <= upper bounds
Initial solution within bounds
SUCCESS: All structures are correctly defined!


**GENERATION $0$: Initialization:**

If our system has **$150$ Optimizable fuzzy parameters** (the $a, b, c, \sigma$ values), then **one vector** (e.g., $\mathbf{x}_1$) is a 150-dimensional array containing one specific set of values for all those parameters.

If our **`population size`** ($N_P$) for the Differential Evolution algorithm is set to **$50$**, then we will define **$50$ such vectors** ($\mathbf{x}_1, \mathbf{x}_2, \dots, \mathbf{x}_{50}$), each representing a unique, complete, candidate solution (a specific design for all $150$ fuzzy membership function parameters). The $DE$ algorithm then works on this entire collection of $50$ vectors simultaneously.

The population size ($N_P$) is simply the number of candidate solution vectors present in the population at any given time. For instance, if we set $N_P=50$, the algorithm works simultaneously with $50$ different 150-dimensional vectors of fuzzy parameters. A larger population size allows $DE$ to explore the vast dimensional search space more thoroughly, increasing the chances of finding the true global optimum, although it also increases the computational time required for each generation.

**`Population Size`:** Let's use **$N_{P} = 50$** individuals (in practice, we'd use $50-100$).

**Step-by-Step Initialization:**

**Individual-1 $(x₁)$:**
```py
  For each parameter i ∈ {1, 2, ..., 150}:
      Generate random value in [lower_bound[i], upper_bound[i]]
```
That is, for every single one of the 150 distinct fuzzy parameters (which could be an $a, b, c, d, c,$ or $\sigma$ value), the algorithm must generate a starting value. It does this by randomly picking a number that falls within a pre-defined, safe, and feasible range specific to that parameter. This process ensures that every parameter within the initial 150-component solution vector is assigned a valid, randomized starting value before the evolution process begins.

For the initialization phase of Differential Evolution, we take the 150 fuzzy parameters that need tuning (e.g., all the $a, b, c, \sigma$ values). For each parameter, we consult the expert-defined boundaries (its unique $\text{lower\_bound}[i]$ and $\text{upper\_bound}[i]$) and generate a random value within that range. We then wrap all $150$ of these randomly generated values into a single vector of size $150$, which represents one complete candidate solution ($\mathbf{x}_i$). Here, we set the population size to $N_P=50$, we repeat this entire process 50 times, generating 50 unique, random, yet boundary-compliant vectors to form our initial population.

So **$x₁$** might look like:
```py
    x₁ = [0, 0, 1.2, 5.7, 1.8, 6.3, 9.5, 2.1, 5.4, 3.8, ..., (150 values total)]
```

**Similarly, generate $x₂, x₃, ..., x₁₀, .....$:**

```py
    x₂ = [0, 0, 9.5, 1.1, 1.3, 5.9, 0.2, 1.8, 6.1, 3.3, ...]

    x₃ = [0, 0, 1.1, 6.2, 1.5, 1.8, 9.1, 0.5, 2.8, 1.5, ...]

    ...
    
    x₁₀ = [0, 0, 1.3, 1.9, 1.1, 6.5, 1.8, 0.2, 2.3, 2.1, ...]
    
    ....
    
    ....50th 
```

**Evaluate Fitness for Each Individual:**   
For each **$x_i$**, we need to:
   1. Use its parameters to fuzzify the dataset
   
   2. Train a Random Forest on the fuzzified features
   
   3. Evaluate performance (`accuracy`, `F1-score`, etc.)

--------------------
-----------------
-----------------

## Step 1: Import Libraries and Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import triang, trapezoid
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

Libraries imported successfully!
NumPy version: 2.3.5
Pandas version: 2.3.3



## Step 2: Define Fuzzy Membership Functions

In [2]:
def trapezoid_membership(x, a, b, c, d):
    """
    Trapezoidal membership function.
    Full membership between b and c, linear transitions at edges.
    """
    return np.maximum(0, np.minimum(1, np.minimum((x - a) / (b - a + 1e-10), 
                                                    (d - x) / (d - c + 1e-10))))

def triangular_membership(x, a, b, c):
    """
    Triangular membership function.
    Peak at b, linear transitions on both sides.
    """
    return np.maximum(0, np.minimum((x - a) / (b - a + 1e-10), 
                                     (c - x) / (c - b + 1e-10)))

print("Membership functions defined!")
print("Testing trapezoid_membership(5, 2, 4, 6, 8):", trapezoid_membership(5, 2, 4, 6, 8))
print("Testing triangular_membership(5, 3, 5, 7):", triangular_membership(5, 3, 5, 7))

Membership functions defined!
Testing trapezoid_membership(5, 2, 4, 6, 8): 1.0
Testing triangular_membership(5, 3, 5, 7): 0.99999999995



## Step 3: Define Solution Vector Structure


In [3]:
def create_initial_solution_vector():
    """
    Creates the initial solution vector with expert-defined parameter values.
    Total: 150 parameters (99 continuous + 51 discrete)
    """
    solution = []
    
    # 1. AGE [Domain: 16-44] - 7 params
    solution.extend([22, 28])  # Young Mother trap c, d
    solution.extend([26, 30, 38])  # Prime Motherhood tri a, b, c
    solution.extend([36, 40])  # Advanced Age trap a, b
    
    # 2. MONTHLY HOUSEHOLD INCOME [Domain: 8000-180000] - 7 params
    solution.extend([20000, 45000])  # Low Income trap b, d
    solution.extend([30000, 50000, 100000])  # Moderate Income tri
    solution.extend([80000, 120000])  # High Income trap a, b
    
    # 3. HOUSING QUALITY [Domain: 1-10] - 7 params
    solution.extend([3.0, 6.0])  # Low Quality trap b, d
    solution.extend([4.0, 6.0, 8.0])  # Moderate Quality tri
    solution.extend([7.0, 9.0])  # High Quality trap a, b
    
    # 4. FINANCIAL STRESS LEVEL [Domain: 0-10] - 7 params
    solution.extend([2.0, 4.0])  # Low Stress trap b, d
    solution.extend([3.0, 5.0, 7.0])  # Moderate Stress tri
    solution.extend([6.0, 8.0])  # High Stress trap a, b
    
    # 5. PARITY [Domain: 0-8] - 7 params
    solution.extend([1.0, 3.0])  # Low Parity trap b, d
    solution.extend([1.0, 2.0, 4.0])  # Optimal Parity tri
    solution.extend([3.0, 5.0])  # High Parity trap a, b
    
    # 6. ANC VISITS [Domain: 0-12] - 7 params
    solution.extend([3.0, 5.0])  # Inadequate ANC trap b, d
    solution.extend([3.0, 5.0, 8.0])  # Sufficient ANC tri
    solution.extend([7.0, 9.0])  # Optimal ANC trap a, b
    
    # 7. GESTATIONAL AGE [Domain: 29-42] - 8 params
    solution.extend([36.0, 38.0])  # Preterm trap b, d
    solution.extend([37.0, 38.0, 41.0, 42.0])  # Optimal Term trap (full)
    solution.extend([40.0, 42.0])  # Late/Post-term trap a, b
    
    # 8. MATERNAL AGE AT FIRST BIRTH [Domain: 15-44] - 8 params
    solution.extend([19.0, 22.0])  # Early Motherhood trap b, d
    solution.extend([20.0, 21.0, 33.0, 35.0])  # Optimal Age trap (full)
    solution.extend([34.0, 35.0])  # Advanced Age trap a, b
    
    # 9. PARTNER SUPPORT SCORE [Domain: 0-10] - 7 params
    solution.extend([4.0, 6.0])  # Low Support trap b, d
    solution.extend([4.0, 6.0, 8.0])  # Moderate Support tri
    solution.extend([7.0, 9.0])  # High Support trap a, b
    
    # 10. FAMILY SUPPORT SCORE [Domain: 0-10] - 7 params
    solution.extend([4.0, 6.0])  # Inadequate Support trap b, d
    solution.extend([4.0, 6.0, 8.0])  # Moderate Support tri
    solution.extend([7.0, 9.0])  # Strong Support trap a, b
    
    # 11. SOCIAL ISOLATION SCORE [Domain: 0-10] - 7 params
    solution.extend([2.0, 4.0])  # Low Isolation trap b, d
    solution.extend([3.0, 4.5, 7.0])  # Moderate Isolation tri
    solution.extend([6.0, 8.0])  # High Isolation trap a, b
    
    # 12. SLEEP QUALITY [Domain: 1-10] - 7 params
    solution.extend([4.0, 6.0])  # Poor Sleep trap b, d
    solution.extend([4.0, 6.5, 8.0])  # Moderate Sleep tri
    solution.extend([7.0, 9.0])  # Good Sleep trap a, b
    
    # 13. STRESSFUL LIFE EVENTS [Domain: 0-5] - 6 params
    solution.extend([1.5])  # Low Stress Load trap d only
    solution.extend([0.5, 2.0, 3.5])  # Moderate Stress Load tri
    solution.extend([3.0, 4.0])  # High Stress Load trap a, b
    
    # 14. DISTANCE TO HEALTH FACILITY [Domain: 0.5-50] - 7 params
    solution.extend([2.0, 5.0])  # Easy Access trap b, d
    solution.extend([3.0, 6.0, 10.0])  # Moderate Access tri
    solution.extend([8.0, 15.0])  # Difficult Access trap a, b
    
    # DISCRETE FEATURES (51 parameters)
    
    # 15. EDUCATION LEVEL - 12 params
    solution.extend([
        1.0, 0.0, 0.0,  # No education
        0.7, 0.3, 0.0,  # Primary
        0.2, 0.8, 0.0,  # Secondary
        0.0, 0.1, 0.9   # Higher
    ])
    
    # 16. WEALTH INDEX - 15 params
    solution.extend([
        1.0, 0.0, 0.0,  # Poorest
        0.7, 0.3, 0.0,  # Poorer
        0.3, 0.7, 0.0,  # Middle
        0.0, 0.3, 0.7,  # Richer
        0.0, 0.0, 1.0   # Richest
    ])
    
    # 17. FOOD SECURITY - 12 params
    solution.extend([
        1.0, 0.0, 0.0,  # Severely insecure
        0.8, 0.2, 0.0,  # Moderately insecure
        0.3, 0.7, 0.0,  # Mildly insecure
        0.0, 0.0, 1.0   # Secure
    ])
    
    # 18. DOMESTIC VIOLENCE EXPOSURE - 12 params
    solution.extend([
        1.0, 0.0, 0.0,  # Often
        0.9, 0.1, 0.0,  # Sometimes
        0.3, 0.7, 0.0,  # Rarely
        0.0, 0.0, 1.0   # Never
    ])
    
    return np.array(solution)

# Create and display initial solution
initial_solution = create_initial_solution_vector()
print(f"Initial solution vector created!")
print(f"Total parameters: {len(initial_solution)}")
print(f"First 20 parameters: {initial_solution[:20]}")
print(f"Last 20 parameters: {initial_solution[-20:]}")

Initial solution vector created!
Total parameters: 150
First 20 parameters: [2.2e+01 2.8e+01 2.6e+01 3.0e+01 3.8e+01 3.6e+01 4.0e+01 2.0e+04 4.5e+04
 3.0e+04 5.0e+04 1.0e+05 8.0e+04 1.2e+05 3.0e+00 6.0e+00 4.0e+00 6.0e+00
 8.0e+00 7.0e+00]
Last 20 parameters: [0.2 0.  0.3 0.7 0.  0.  0.  1.  1.  0.  0.  0.9 0.1 0.  0.3 0.7 0.  0.
 0.  1. ]



## Step 4: Define Bounds for DE


In [4]:
def create_de_bounds():
    """
    Creates bounds for all 150 parameters.
    Returns: numpy array of shape (150, 2) with [lower_bound, upper_bound]
    """
    bounds = []
    
    # 1. AGE - 7 params
    bounds.extend([
        [18, 26], [24, 32],  # Young c, d
        [22, 28], [28, 32], [32, 40],  # Prime a, b, c
        [32, 38], [36, 42]  # Advanced a, b
    ])
    
    # 2. INCOME - 7 params
    bounds.extend([
        [15000, 25000], [35000, 55000],  # Low b, d
        [25000, 40000], [40000, 60000], [80000, 120000],  # Moderate
        [70000, 90000], [100000, 140000]  # High a, b
    ])
    
    # 3. HOUSING QUALITY - 7 params
    bounds.extend([
        [2.0, 4.0], [5.0, 7.0],  # Low b, d
        [3.0, 5.0], [5.0, 7.0], [7.0, 9.0],  # Moderate
        [6.0, 8.0], [8.0, 9.5]  # High a, b
    ])
    
    # 4. FINANCIAL STRESS - 7 params
    bounds.extend([
        [1.0, 3.0], [3.0, 5.0],  # Low b, d
        [2.0, 4.0], [4.0, 6.0], [6.0, 8.0],  # Moderate
        [5.0, 7.0], [7.0, 9.0]  # High a, b
    ])
    
    # 5. PARITY - 7 params
    bounds.extend([
        [0.5, 1.5], [2.0, 4.0],  # Low b, d
        [0.5, 1.5], [1.5, 2.5], [3.0, 5.0],  # Optimal
        [2.0, 4.0], [4.0, 6.0]  # High a, b
    ])
    
    # 6. ANC VISITS - 7 params
    bounds.extend([
        [2.0, 4.0], [4.0, 6.0],  # Inadequate b, d
        [2.0, 4.0], [4.0, 6.0], [7.0, 9.0],  # Sufficient
        [6.0, 8.0], [8.0, 10.0]  # Optimal a, b
    ])
    
    # 7. GESTATIONAL AGE - 8 params
    bounds.extend([
        [34.0, 37.0], [37.0, 39.0],  # Preterm b, d
        [36.0, 38.0], [37.0, 39.0], [40.0, 41.5], [41.0, 42.0],  # Optimal
        [39.0, 41.0], [41.0, 42.0]  # Late a, b
    ])
    
    # 8. MATERNAL AGE FIRST BIRTH - 8 params
    bounds.extend([
        [17.0, 20.0], [20.0, 24.0],  # Early b, d
        [18.0, 22.0], [20.0, 24.0], [30.0, 35.0], [33.0, 37.0],  # Optimal
        [32.0, 36.0], [34.0, 38.0]  # Advanced a, b
    ])
    
    # 9. PARTNER SUPPORT - 7 params
    bounds.extend([
        [3.0, 5.0], [5.0, 7.0],  # Low b, d
        [3.0, 5.0], [5.0, 7.0], [7.0, 9.0],  # Moderate
        [6.0, 8.0], [8.0, 9.5]  # High a, b
    ])
    
    # 10. FAMILY SUPPORT - 7 params
    bounds.extend([
        [3.0, 5.0], [5.0, 7.0],  # Inadequate b, d
        [3.0, 5.0], [5.0, 7.0], [7.0, 9.0],  # Moderate
        [6.0, 8.0], [8.0, 9.5]  # Strong a, b
    ])
    
    # 11. SOCIAL ISOLATION - 7 params
    bounds.extend([
        [1.0, 3.0], [3.0, 5.0],  # Low b, d
        [2.0, 4.0], [4.0, 5.5], [6.0, 8.0],  # Moderate
        [5.0, 7.0], [7.0, 9.0]  # High a, b
    ])
    
    # 12. SLEEP QUALITY - 7 params
    bounds.extend([
        [3.0, 5.0], [5.0, 7.0],  # Poor b, d
        [3.0, 5.0], [5.5, 7.5], [7.0, 9.0],  # Moderate
        [6.0, 8.0], [8.0, 9.5]  # Good a, b
    ])
    
    # 13. STRESSFUL LIFE EVENTS - 6 params
    bounds.extend([
        [1.0, 2.0],  # Low d
        [0.3, 0.8], [1.5, 2.5], [3.0, 4.0],  # Moderate
        [2.5, 3.5], [3.5, 4.5]  # High a, b
    ])
    
    # 14. DISTANCE TO FACILITY - 7 params
    bounds.extend([
        [1.5, 2.5], [4.0, 6.0],  # Easy b, d
        [2.5, 4.0], [5.0, 7.0], [8.0, 12.0],  # Moderate
        [7.0, 10.0], [12.0, 18.0]  # Difficult a, b
    ])
    
    # DISCRETE FEATURES (51 parameters) - All membership degrees [0.0, 1.0]
    for _ in range(51):
        bounds.append([0.0, 1.0])
    
    return np.array(bounds)

# Create and display bounds
bounds = create_de_bounds()
print(f"Bounds array created!")
print(f"Shape: {bounds.shape}")
print(f"First 10 bounds:\n{bounds[:10]}")
print(f"Last 10 bounds:\n{bounds[-10:]}")

Bounds array created!
Shape: (150, 2)
First 10 bounds:
[[1.8e+01 2.6e+01]
 [2.4e+01 3.2e+01]
 [2.2e+01 2.8e+01]
 [2.8e+01 3.2e+01]
 [3.2e+01 4.0e+01]
 [3.2e+01 3.8e+01]
 [3.6e+01 4.2e+01]
 [1.5e+04 2.5e+04]
 [3.5e+04 5.5e+04]
 [2.5e+04 4.0e+04]]
Last 10 bounds:
[[0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]]



## Step 5: Parameter Decoder Function


In [5]:
def decode_solution_to_fuzzy_params(solution):
    """
    Decodes solution vector into fuzzy membership function parameters.
    Returns a dictionary mapping feature names to their fuzzy parameters.
    """
    params = {}
    idx = 0
    
    # 1. AGE (7 params)
    params['age'] = {
        'young': [16, 16, solution[idx], solution[idx+1]],
        'prime': [solution[idx+2], solution[idx+3], solution[idx+4]],
        'advanced': [solution[idx+5], solution[idx+6], 44, 44]
    }
    idx += 7
    
    # 2. INCOME (7 params)
    params['income'] = {
        'low': [8000, solution[idx], solution[idx], solution[idx+1]],
        'moderate': [solution[idx+2], solution[idx+3], solution[idx+4]],
        'high': [solution[idx+5], solution[idx+6], 180000, 180000]
    }
    idx += 7
    
    # 3. HOUSING QUALITY (7 params)
    params['housing_quality'] = {
        'low': [1.0, solution[idx], solution[idx], solution[idx+1]],
        'moderate': [solution[idx+2], solution[idx+3], solution[idx+4]],
        'high': [solution[idx+5], solution[idx+6], 10.0, 10.0]
    }
    idx += 7
    
    # 4. FINANCIAL STRESS (7 params)
    params['financial_stress'] = {
        'low': [0.0, solution[idx], solution[idx], solution[idx+1]],
        'moderate': [solution[idx+2], solution[idx+3], solution[idx+4]],
        'high': [solution[idx+5], solution[idx+6], 10.0, 10.0]
    }
    idx += 7
    
    # 5. PARITY (7 params)
    params['parity'] = {
        'low': [0.0, solution[idx], solution[idx], solution[idx+1]],
        'optimal': [solution[idx+2], solution[idx+3], solution[idx+4]],
        'high': [solution[idx+5], solution[idx+6], 8.0, 8.0]
    }
    idx += 7
    
    # 6. ANC VISITS (7 params)
    params['anc_visits'] = {
        'inadequate': [0.0, solution[idx], solution[idx], solution[idx+1]],
        'sufficient': [solution[idx+2], solution[idx+3], solution[idx+4]],
        'optimal': [solution[idx+5], solution[idx+6], 12.0, 12.0]
    }
    idx += 7
    
    # 7. GESTATIONAL AGE (8 params)
    params['gestational_age'] = {
        'preterm': [29.0, solution[idx], solution[idx], solution[idx+1]],
        'optimal': [solution[idx+2], solution[idx+3], solution[idx+4], solution[idx+5]],
        'late_post': [solution[idx+6], solution[idx+7], 42.0, 42.0]
    }
    idx += 8
    
    # 8. MATERNAL AGE FIRST BIRTH (8 params)
    params['maternal_age_first_birth'] = {
        'early': [15.0, solution[idx], solution[idx], solution[idx+1]],
        'optimal': [solution[idx+2], solution[idx+3], solution[idx+4], solution[idx+5]],
        'advanced': [solution[idx+6], solution[idx+7], 44.0, 44.0]
    }
    idx += 8
    
    # 9. PARTNER SUPPORT (7 params)
    params['partner_support'] = {
        'low': [0.0, solution[idx], solution[idx], solution[idx+1]],
        'moderate': [solution[idx+2], solution[idx+3], solution[idx+4]],
        'high': [solution[idx+5], solution[idx+6], 10.0, 10.0]
    }
    idx += 7
    
    # 10. FAMILY SUPPORT (7 params)
    params['family_support'] = {
        'inadequate': [0.0, solution[idx], solution[idx], solution[idx+1]],
        'moderate': [solution[idx+2], solution[idx+3], solution[idx+4]],
        'strong': [solution[idx+5], solution[idx+6], 10.0, 10.0]
    }
    idx += 7
    
    # 11. SOCIAL ISOLATION (7 params)
    params['social_isolation'] = {
        'low': [0.0, solution[idx], solution[idx], solution[idx+1]],
        'moderate': [solution[idx+2], solution[idx+3], solution[idx+4]],
        'high': [solution[idx+5], solution[idx+6], 10.0, 10.0]
    }
    idx += 7
    
    # 12. SLEEP QUALITY (7 params)
    params['sleep_quality'] = {
        'poor': [1.0, solution[idx], solution[idx], solution[idx+1]],
        'moderate': [solution[idx+2], solution[idx+3], solution[idx+4]],
        'good': [solution[idx+5], solution[idx+6], 10.0, 10.0]
    }
    idx += 7
    
    # 13. STRESSFUL LIFE EVENTS (6 params)
    params['stressful_life_events'] = {
        'low': [0.0, 0.0, 0.0, solution[idx]],
        'moderate': [solution[idx+1], solution[idx+2], solution[idx+3]],
        'high': [solution[idx+4], solution[idx+5], 5.0, 5.0]
    }
    idx += 6
    
    # 14. DISTANCE TO FACILITY (7 params)
    params['distance_to_facility'] = {
        'easy': [0.5, solution[idx], solution[idx], solution[idx+1]],
        'moderate': [solution[idx+2], solution[idx+3], solution[idx+4]],
        'difficult': [solution[idx+5], solution[idx+6], 50.0, 50.0]
    }
    idx += 7
    
    # DISCRETE FEATURES (51 params)
    
    # 15. EDUCATION (12 params)
    params['education'] = {
        'No education': solution[idx:idx+3],
        'Primary': solution[idx+3:idx+6],
        'Secondary': solution[idx+6:idx+9],
        'Higher': solution[idx+9:idx+12]
    }
    idx += 12
    
    # 16. WEALTH INDEX (15 params)
    params['wealth'] = {
        'Poorest': solution[idx:idx+3],
        'Poorer': solution[idx+3:idx+6],
        'Middle': solution[idx+6:idx+9],
        'Richer': solution[idx+9:idx+12],
        'Richest': solution[idx+12:idx+15]
    }
    idx += 15
    
    # 17. FOOD SECURITY (12 params)
    params['food_security'] = {
        'Severely insecure': solution[idx:idx+3],
        'Moderately insecure': solution[idx+3:idx+6],
        'Mildly insecure': solution[idx+6:idx+9],
        'Secure': solution[idx+9:idx+12]
    }
    idx += 12
    
    # 18. DOMESTIC VIOLENCE (12 params)
    params['domestic_violence'] = {
        'Often': solution[idx:idx+3],
        'Sometimes': solution[idx+3:idx+6],
        'Rarely': solution[idx+6:idx+9],
        'Never': solution[idx+9:idx+12]
    }
    
    return params

# Test decoder
test_params = decode_solution_to_fuzzy_params(initial_solution)
print("Decoder test successful!")
print(f"Number of features decoded: {len(test_params)}")
print(f"\nAge parameters: {test_params['age']}")
print(f"\nEducation parameters: {test_params['education']}")

Decoder test successful!
Number of features decoded: 18

Age parameters: {'young': [16, 16, np.float64(22.0), np.float64(28.0)], 'prime': [np.float64(26.0), np.float64(30.0), np.float64(38.0)], 'advanced': [np.float64(36.0), np.float64(40.0), 44, 44]}

Education parameters: {'No education': array([1., 0., 0.]), 'Primary': array([0.7, 0.3, 0. ]), 'Secondary': array([0.2, 0.8, 0. ]), 'Higher': array([0. , 0.1, 0.9])}



## Step 6: Apply Fuzzy Transformation to Dataset


In [6]:
def apply_fuzzy_transformation(df_raw, solution):
    """
    Applies fuzzy transformation to raw data using parameters from solution vector.
    Assumes df_raw contains the original feature columns.
    """
    df = df_raw.copy()
    params = decode_solution_to_fuzzy_params(solution)
    
    # Mapping of raw column names to parameter keys
    # You'll need to adjust these based on your actual column names
    column_mapping = {
        'age': 'age',
        'monthly_household_income': 'income',
        'housing_quality': 'housing_quality',
        'financial_stress_level': 'financial_stress',
        'parity': 'parity',
        'anc_visits': 'anc_visits',
        'gestational_age': 'gestational_age',
        'maternal_age_at_first_birth': 'maternal_age_first_birth',
        'partner_support_score': 'partner_support',
        'family_support_score': 'family_support',
        'social_isolation_score': 'social_isolation',
        'sleep_quality': 'sleep_quality',
        'stressful_life_events': 'stressful_life_events',
        'distance_to_health_facility': 'distance_to_facility',
        'education_level': 'education',
        'wealth_index': 'wealth',
        'food_security': 'food_security',
        'domestic_violence_exposure': 'domestic_violence'
    }
    
    # Apply continuous feature transformations
    continuous_features = [
        ('age', ['young', 'prime', 'advanced']),
        ('income', ['low', 'moderate', 'high']),
        ('housing_quality', ['low', 'moderate', 'high']),
        ('financial_stress', ['low', 'moderate', 'high']),
        ('parity', ['low', 'optimal', 'high']),
        ('anc_visits', ['inadequate', 'sufficient', 'optimal']),
        ('gestational_age', ['preterm', 'optimal', 'late_post']),
        ('maternal_age_first_birth', ['early', 'optimal', 'advanced']),
        ('partner_support', ['low', 'moderate', 'high']),
        ('family_support', ['inadequate', 'moderate', 'strong']),
        ('social_isolation', ['low', 'moderate', 'high']),
        ('sleep_quality', ['poor', 'moderate', 'good']),
        ('stressful_life_events', ['low', 'moderate', 'high']),
        ('distance_to_facility', ['easy', 'moderate', 'difficult'])
    ]
    
    for raw_col, param_key in column_mapping.items():
        if raw_col not in df.columns:
            continue
            
        if param_key in ['education', 'wealth', 'food_security', 'domestic_violence']:
            # Discrete features
            for category, memberships in params[param_key].items():
                mask = df[raw_col] == category
                if param_key == 'education':
                    df.loc[mask, 'mu_edu_low_risk'] = memberships[0]
                    df.loc[mask, 'mu_edu_medium_risk'] = memberships[1]
                    df.loc[mask, 'mu_edu_high_risk'] = memberships[2]
                elif param_key == 'wealth':
                    df.loc[mask, 'mu_wealth_high_poverty'] = memberships[0]
                    df.loc[mask, 'mu_wealth_medium_wealth'] = memberships[1]
                    df.loc[mask, 'mu_wealth_low_poverty'] = memberships[2]
                elif param_key == 'food_security':
                    df.loc[mask, 'mu_fs_high_risk'] = memberships[0]
                    df.loc[mask, 'mu_fs_mod_risk'] = memberships[1]
                    df.loc[mask, 'mu_fs_low_risk'] = memberships[2]
                elif param_key == 'domestic_violence':
                    df.loc[mask, 'mu_dv_high_risk'] = memberships[0]
                    df.loc[mask, 'mu_dv_mod_risk'] = memberships[1]
                    df.loc[mask, 'mu_dv_no_risk'] = memberships[2]
        else:
            # Continuous features
            feature_params = params[param_key]
            values = df[raw_col].values
            
            for fuzzy_set, set_params in feature_params.items():
                if len(set_params) == 3:  # Triangular
                    membership = triangular_membership(values, *set_params)
                else:  # Trapezoidal
                    membership = trapezoid_membership(values, *set_params)
                
                # Create column name based on feature
                col_name = create_fuzzy_column_name(param_key, fuzzy_set)
                df[col_name] = membership
    
    return df

def create_fuzzy_column_name(feature, fuzzy_set):
    """Helper to create standardized fuzzy column names"""
    mapping = {
        'age': {'young': 'mu_age_young', 'prime': 'mu_age_prime', 'advanced': 'mu_age_advanced'},
        'income': {'low': 'mu_inc_low', 'moderate': 'mu_inc_mod', 'high': 'mu_inc_high'},
        'housing_quality': {'low': 'mu_hq_low', 'moderate': 'mu_hq_mod', 'high': 'mu_hq_high'},
        'financial_stress': {'low': 'mu_stress_low', 'moderate': 'mu_stress_mod', 'high': 'mu_stress_high'},
        'parity': {'low': 'mu_parity_low', 'optimal': 'mu_parity_opt', 'high': 'mu_parity_high'},
        'anc_visits': {'inadequate': 'mu_anc_inadequate', 'sufficient': 'mu_anc_sufficient', 'optimal': 'mu_anc_optimal'},
        'gestational_age': {'preterm': 'mu_ga_preterm', 'optimal': 'mu_ga_optimal', 'late_post': 'mu_ga_late_post'},
        'maternal_age_first_birth': {'early': 'mu_maternal_age_early', 'optimal': 'mu_maternal_age_optimal', 'advanced': 'mu_maternal_age_advanced'},
        'partner_support': {'low': 'mu_support_low', 'moderate': 'mu_support_mod', 'high': 'mu_support_high'},
        'family_support': {'inadequate': 'mu_family_inadequate', 'moderate': 'mu_family_moderate', 'strong': 'mu_family_strong'},
        'social_isolation': {'low': 'mu_isolation_low', 'moderate': 'mu_isolation_mod', 'high': 'mu_isolation_high'},
        'sleep_quality': {'poor': 'mu_sleep_poor', 'moderate': 'mu_sleep_moderate', 'good': 'mu_sleep_good'},
        'stressful_life_events': {'low': 'mu_stress_low', 'moderate': 'mu_stress_mod', 'high': 'mu_stress_high'},
        'distance_to_facility': {'easy': 'mu_distance_easy', 'moderate': 'mu_distance_moderate', 'difficult': 'mu_distance_difficult'}
    }
    return mapping[feature][fuzzy_set]

print("Fuzzy transformation function defined!")
print("Ready to transform datasets with optimized parameters.")

Fuzzy transformation function defined!
Ready to transform datasets with optimized parameters.



Now I need to see your actual dataset structure to continue. Could you please run:

In [12]:
df= pd.read_csv('C:\\Users\MyMachine\\Desktop\Fuzzy-Logic-Based-Maternal-Mental-Health-Risk-Assessment\\Data_Synthesis\\sampled_maternal_data_20k.csv')

In [13]:
df.columns

Index(['age', 'residence', 'province', 'ethnicity',
       'caste_discrimination_exposure', 'education_level', 'occupation',
       'family_type', 'wealth_index', 'monthly_household_income',
       'food_security', 'housing_quality', 'financial_stress_level', 'parity',
       'anc_visits', 'delivery_type', 'birth_complications',
       'baby_health_status', 'gestational_age', 'maternal_age_at_first_birth',
       'previous_pregnancy_loss', 'partner_support_score',
       'family_support_score', 'domestic_violence_exposure',
       'social_isolation_score', 'sleep_quality', 'stressful_life_events',
       'mental_health_awareness', 'distance_to_health_facility',
       'health_insurance', 'previous_mental_health_consultation',
       'ppd_risk_level'],
      dtype='object')

In [14]:
# Display basic info about your dataset
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nTarget variable (if exists):")
print(df['ppd_risk_level'].value_counts())

Dataset shape: (20000, 32)

Column names:
['age', 'residence', 'province', 'ethnicity', 'caste_discrimination_exposure', 'education_level', 'occupation', 'family_type', 'wealth_index', 'monthly_household_income', 'food_security', 'housing_quality', 'financial_stress_level', 'parity', 'anc_visits', 'delivery_type', 'birth_complications', 'baby_health_status', 'gestational_age', 'maternal_age_at_first_birth', 'previous_pregnancy_loss', 'partner_support_score', 'family_support_score', 'domestic_violence_exposure', 'social_isolation_score', 'sleep_quality', 'stressful_life_events', 'mental_health_awareness', 'distance_to_health_facility', 'health_insurance', 'previous_mental_health_consultation', 'ppd_risk_level']

First few rows:
   age residence       province        ethnicity  \
0   21     Urban          Koshi  Hill High Caste   
1   38     Rural        Madhesh            Dalit   
2   38     Rural  Sudurpashchim      Terai Caste   
3   41     Rural  Sudurpashchim         Janajati   
4  

In [ ]:
Perfect! Now I understand. The dataset has 32 original columns, and from these, fuzzy columns were created. Let me continue with the DE implementation.

## Step 7: Fitness Function with Random Forest


In [15]:
def evaluate_solution(solution, df_raw, fuzzy_feature_cols, other_feature_cols, target_col='ppd_risk_level', cv_folds=5):
    """
    Evaluates a solution vector by:
    1. Applying fuzzy transformation with solution parameters
    2. Training Random Forest with cross-validation
    3. Returning mean CV accuracy
    
    Parameters:
    - solution: 150-parameter vector
    - df_raw: Original dataframe with raw features
    - fuzzy_feature_cols: List of fuzzy column names to use
    - other_feature_cols: List of non-fuzzy features to include
    - target_col: Target variable name
    - cv_folds: Number of CV folds
    
    Returns:
    - fitness: Mean cross-validation accuracy (higher is better)
    """
    try:
        # Apply fuzzy transformation
        df_transformed = apply_fuzzy_transformation(df_raw, solution)
        
        # Select features and target
        all_features = fuzzy_feature_cols + other_feature_cols
        X = df_transformed[all_features].values
        y = df_transformed[target_col].values
        
        # Handle any NaN values that might arise from transformation
        if np.any(np.isnan(X)):
            return 0.0  # Invalid solution
        
        # Random Forest with cross-validation
        rf = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_split=10,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )
        
        # Perform cross-validation
        cv_scores = cross_val_score(rf, X, y, cv=cv_folds, scoring='accuracy', n_jobs=-1)
        fitness = np.mean(cv_scores)
        
        return fitness
    
    except Exception as e:
        print(f"Error in evaluation: {e}")
        return 0.0

print("Fitness function defined!")

Fitness function defined!


In [ ]:

## Step 8: Define Fuzzy and Other Feature Columns


In [16]:
# Define the 54 fuzzy feature columns (as per your description)
fuzzy_feature_cols = [
    'mu_age_young', 'mu_age_prime', 'mu_age_advanced',
    'mu_edu_low_risk', 'mu_edu_medium_risk', 'mu_edu_high_risk',
    'mu_wealth_high_poverty', 'mu_wealth_medium_wealth', 'mu_wealth_low_poverty',
    'mu_inc_low', 'mu_inc_mod', 'mu_inc_high',
    'mu_fs_high_risk', 'mu_fs_mod_risk', 'mu_fs_low_risk',
    'mu_hq_low', 'mu_hq_mod', 'mu_hq_high',
    'mu_stress_low', 'mu_stress_mod', 'mu_stress_high',
    'mu_parity_low', 'mu_parity_opt', 'mu_parity_high',
    'mu_anc_inadequate', 'mu_anc_sufficient', 'mu_anc_optimal',
    'mu_ga_preterm', 'mu_ga_optimal', 'mu_ga_late_post',
    'mu_maternal_age_early', 'mu_maternal_age_optimal', 'mu_maternal_age_advanced',
    'mu_support_low', 'mu_support_mod', 'mu_support_high',
    'mu_family_inadequate', 'mu_family_moderate', 'mu_family_strong',
    'mu_dv_high_risk', 'mu_dv_mod_risk', 'mu_dv_no_risk',
    'mu_isolation_low', 'mu_isolation_mod', 'mu_isolation_high',
    'mu_sleep_poor', 'mu_sleep_moderate', 'mu_sleep_good',
    'mu_stress_low', 'mu_stress_mod', 'mu_stress_high',
    'mu_distance_easy', 'mu_distance_moderate', 'mu_distance_difficult'
]

# Define other features that are NOT fuzzified (you can adjust this list)
other_feature_cols = [
    'residence', 'province', 'ethnicity', 'caste_discrimination_exposure',
    'occupation', 'family_type', 'delivery_type', 'birth_complications',
    'baby_health_status', 'previous_pregnancy_loss', 'mental_health_awareness',
    'health_insurance', 'previous_mental_health_consultation'
]

# For categorical features, we'll need to encode them
# Let's create an encoding function

def prepare_features(df, fuzzy_cols, other_cols):
    """
    Prepares feature matrix by encoding categorical variables
    """
    df_encoded = df.copy()
    
    # One-hot encode categorical features
    categorical_cols = df[other_cols].select_dtypes(include=['object']).columns.tolist()
    
    if categorical_cols:
        df_encoded = pd.get_dummies(df_encoded, columns=categorical_cols, drop_first=True)
    
    # Get updated other_cols after encoding
    encoded_other_cols = [col for col in df_encoded.columns if col.startswith(tuple(other_cols))]
    
    return df_encoded, fuzzy_cols, encoded_other_cols

print(f"Number of fuzzy features: {len(fuzzy_feature_cols)}")
print(f"Number of other features: {len(other_feature_cols)}")
print("\nFeature preparation function defined!")

Number of fuzzy features: 54
Number of other features: 13

Feature preparation function defined!


In [ ]:

## Step 9: Updated Fitness Function with Encoding


In [17]:
def evaluate_solution_v2(solution, df_raw, fuzzy_cols, other_cols, target_col='ppd_risk_level', cv_folds=5):
    """
    Enhanced evaluation function with proper categorical encoding
    """
    try:
        # Apply fuzzy transformation
        df_transformed = apply_fuzzy_transformation(df_raw, solution)
        
        # Prepare features with encoding
        df_encoded, fuzzy_cols_final, other_cols_final = prepare_features(
            df_transformed, fuzzy_cols, other_cols
        )
        
        # Select features and target
        all_features = fuzzy_cols_final + other_cols_final
        
        # Ensure all required columns exist
        missing_cols = [col for col in all_features if col not in df_encoded.columns]
        if missing_cols:
            print(f"Warning: Missing columns: {missing_cols[:5]}...")
            return 0.0
        
        X = df_encoded[all_features].values
        y = df_encoded[target_col].values
        
        # Handle any NaN values
        if np.any(np.isnan(X)):
            nan_count = np.sum(np.isnan(X))
            print(f"Warning: {nan_count} NaN values detected")
            return 0.0
        
        # Random Forest with cross-validation
        rf = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_split=10,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )
        
        # Perform cross-validation
        cv_scores = cross_val_score(rf, X, y, cv=cv_folds, scoring='accuracy', n_jobs=-1)
        fitness = np.mean(cv_scores)
        
        return fitness
    
    except Exception as e:
        print(f"Error in evaluation: {str(e)[:100]}")
        import traceback
        traceback.print_exc()
        return 0.0

print("Enhanced fitness function defined!")

Enhanced fitness function defined!


In [ ]:

## Step 10: Differential Evolution Implementation


In [18]:
class DifferentialEvolution:
    """
    Differential Evolution optimizer for fuzzy parameter tuning
    """
    def __init__(self, bounds, population_size=50, F=0.8, CR=0.9, seed=42):
        """
        Parameters:
        - bounds: Array of shape (n_params, 2) with [lower, upper] bounds
        - population_size: Number of candidate solutions
        - F: Differential weight [0, 2]
        - CR: Crossover probability [0, 1]
        - seed: Random seed
        """
        self.bounds = bounds
        self.n_params = len(bounds)
        self.population_size = population_size
        self.F = F
        self.CR = CR
        self.rng = np.random.RandomState(seed)
        
        # Initialize population
        self.population = self._initialize_population()
        self.fitness = np.zeros(population_size)
        
        # Track best solution
        self.best_solution = None
        self.best_fitness = -np.inf
        
        # History
        self.history = {
            'best_fitness': [],
            'mean_fitness': [],
            'std_fitness': []
        }
    
    def _initialize_population(self):
        """
        Initialize population within bounds using Latin Hypercube Sampling
        """
        population = np.zeros((self.population_size, self.n_params))
        
        for i in range(self.n_params):
            lower, upper = self.bounds[i]
            population[:, i] = self.rng.uniform(lower, upper, self.population_size)
        
        return population
    
    def _mutate(self, idx):
        """
        Create mutant vector using DE/rand/1 strategy
        """
        # Select 3 random distinct individuals (different from idx)
        candidates = [i for i in range(self.population_size) if i != idx]
        r1, r2, r3 = self.rng.choice(candidates, 3, replace=False)
        
        # Mutation: mutant = x_r1 + F * (x_r2 - x_r3)
        mutant = self.population[r1] + self.F * (self.population[r2] - self.population[r3])
        
        # Ensure bounds
        mutant = np.clip(mutant, self.bounds[:, 0], self.bounds[:, 1])
        
        return mutant
    
    def _crossover(self, target, mutant):
        """
        Binomial crossover
        """
        trial = np.copy(target)
        j_rand = self.rng.randint(0, self.n_params)
        
        for j in range(self.n_params):
            if self.rng.rand() < self.CR or j == j_rand:
                trial[j] = mutant[j]
        
        return trial
    
    def evolve(self, fitness_func, generations=10, verbose=True):
        """
        Run the evolutionary process
        """
        print(f"Starting Differential Evolution...")
        print(f"Population size: {self.population_size}")
        print(f"Parameters: {self.n_params}")
        print(f"Generations: {generations}")
        print(f"F={self.F}, CR={self.CR}")
        print("="*60)
        
        # Evaluate initial population
        print("\nEvaluating initial population...")
        for i in range(self.population_size):
            self.fitness[i] = fitness_func(self.population[i])
            if (i + 1) % 10 == 0:
                print(f"  Evaluated {i+1}/{self.population_size} individuals")
        
        # Update best
        best_idx = np.argmax(self.fitness)
        self.best_fitness = self.fitness[best_idx]
        self.best_solution = self.population[best_idx].copy()
        
        print(f"\nInitial Best Fitness: {self.best_fitness:.6f}")
        print(f"Initial Mean Fitness: {np.mean(self.fitness):.6f}")
        print(f"Initial Std Fitness: {np.std(self.fitness):.6f}")
        
        # Evolution loop
        for gen in range(generations):
            print(f"\n{'='*60}")
            print(f"Generation {gen + 1}/{generations}")
            print(f"{'='*60}")
            
            new_population = np.copy(self.population)
            new_fitness = np.copy(self.fitness)
            
            improvements = 0
            
            for i in range(self.population_size):
                # Mutation
                mutant = self._mutate(i)
                
                # Crossover
                trial = self._crossover(self.population[i], mutant)
                
                # Evaluation
                trial_fitness = fitness_func(trial)
                
                # Selection
                if trial_fitness > self.fitness[i]:
                    new_population[i] = trial
                    new_fitness[i] = trial_fitness
                    improvements += 1
                    
                    # Update global best
                    if trial_fitness > self.best_fitness:
                        self.best_fitness = trial_fitness
                        self.best_solution = trial.copy()
                        print(f"  → New best fitness: {self.best_fitness:.6f} (individual {i})")
                
                if (i + 1) % 10 == 0:
                    print(f"  Evaluated {i+1}/{self.population_size} trials")
            
            # Update population
            self.population = new_population
            self.fitness = new_fitness
            
            # Record history
            self.history['best_fitness'].append(self.best_fitness)
            self.history['mean_fitness'].append(np.mean(self.fitness))
            self.history['std_fitness'].append(np.std(self.fitness))
            
            # Print generation summary
            print(f"\nGeneration {gen + 1} Summary:")
            print(f"  Best Fitness: {self.best_fitness:.6f}")
            print(f"  Mean Fitness: {np.mean(self.fitness):.6f}")
            print(f"  Std Fitness: {np.std(self.fitness):.6f}")
            print(f"  Improvements: {improvements}/{self.population_size}")
            print(f"  Improvement Rate: {100*improvements/self.population_size:.1f}%")
        
        print(f"\n{'='*60}")
        print("Evolution Complete!")
        print(f"Final Best Fitness: {self.best_fitness:.6f}")
        print(f"{'='*60}")
        
        return self.best_solution, self.best_fitness

print("Differential Evolution class defined!")

Differential Evolution class defined!


In [ ]:

## Step 11: Initialize and Run DE


In [19]:
# Create bounds
bounds = create_de_bounds()

# Create fitness function wrapper
def fitness_wrapper(solution):
    return evaluate_solution_v2(
        solution=solution,
        df_raw=df,
        fuzzy_cols=fuzzy_feature_cols,
        other_cols=other_feature_cols,
        target_col='ppd_risk_level',
        cv_folds=5
    )

# Test with initial solution first
print("Testing fitness function with initial solution...")
initial_fitness = fitness_wrapper(initial_solution)
print(f"Initial solution fitness: {initial_fitness:.6f}")

Testing fitness function with initial solution...
Error in evaluation: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any s
Initial solution fitness: 0.000000


Traceback (most recent call last):
  File "C:\Users\MyMachine\AppData\Local\Temp\ipykernel_7620\2029425589.py", line 27, in evaluate_solution_v2
    if np.any(np.isnan(X)):
              ~~~~~~~~^^^
TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''


In [ ]:

Please run this cell first to verify that the fitness function works correctly with your data. Once this succeeds, we'll proceed with the full DE optimization.